<a href="https://colab.research.google.com/github/keduog/EDU-AI-Training/blob/main/Day5/Session2/session2_controlling_the_model.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 2 — Controlling the Model

**Day 5 · 10:45 – 12:45**

Anyone can send a prompt. Getting **reliable, well-shaped** answers back is the skill.

## Cell 1 — Setup (same as Session 1)

In [ ]:
!pip install -q -U google-genai

import os
from google import genai
from google.genai import types

# Load the key from Colab Secrets (key icon in the left sidebar)
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    # Not on Colab, or secret not set - fall back to typing it in
    if not os.environ.get("GEMINI_API_KEY"):
        import getpass
        os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")

client = genai.Client()          # reads GEMINI_API_KEY from the environment
MODEL = "gemini-3.5-flash"       # see the model-listing cell if this name fails

print("Client ready. Key length:", len(os.environ["GEMINI_API_KEY"]))

## Cell 2 — System instructions

A system instruction sets the model's role and rules. It applies to **every** message,
and the user never sees it.

This is the single highest-leverage line you will write today.

In [ ]:
config = types.GenerateContentConfig(
    system_instruction=(
        "You are a helpful assistant for the Ethiopian Defense University. "
        "Answer in clear, simple language. "
        "If the question is asked in Amharic, reply in Amharic. "
        "Keep answers under 80 words. "
        "If you do not know something, say so plainly - never invent facts."
    ),
    max_output_tokens=300,
)

for q in ["What is machine learning?", "የኢትዮጵያ ዋና ከተማ ማን ናት?"]:
    r = client.models.generate_content(model=MODEL, contents=q, config=config)
    print("Q:", q)
    print("A:", r.text)
    print("-" * 60)

## Cell 3 — The parameter that no longer works

Almost every tutorial online tells you to control creativity with `temperature`.

**On Gemini 3.x models that is no longer true.** Google deprecated `temperature`,
`top_p` and `top_k` — they are now *ignored*, and will return an error in future models.

> Source: [ai.google.dev/gemini-api/docs/latest-model](https://ai.google.dev/gemini-api/docs/latest-model)
> — *"Sampling parameter deprecation"*

Run this and see for yourself.

In [ ]:
prompt = "Invent a name for a new Ethiopian technology company."

print("Asking the same question 3 times:\n")
for i in range(3):
    r = client.models.generate_content(model=MODEL, contents=prompt)
    print(f"  {i+1}. {r.text.strip()[:80]}")

print()
print("Variation comes from the model itself, not from a temperature setting.")

### What replaced it: writing the rule in words

Instead of a number, describe the behaviour you want. This works better, and it
transfers to every other provider.

In [ ]:
precise = types.GenerateContentConfig(
    system_instruction=(
        "Answer with exactly one short factual sentence. "
        "No preamble, no elaboration, no speculation."))

creative = types.GenerateContentConfig(
    system_instruction=(
        "You are an imaginative writer. Answer with a vivid, "
        "unexpected image. Take creative risks."))

q = "Describe the Simien Mountains."

print("PRECISE:\n ", client.models.generate_content(
    model=MODEL, contents=q, config=precise).text.strip())
print()
print("CREATIVE:\n ", client.models.generate_content(
    model=MODEL, contents=q, config=creative).text.strip())

## Cell 4 — Streaming

Waiting 8 seconds in silence feels broken. Streaming shows words as they arrive.

Same total time — completely different experience.

In [ ]:
stream = client.models.generate_content_stream(
    model=MODEL,
    contents="Explain in three short paragraphs how a satellite stays in orbit.",
)

for chunk in stream:
    if chunk.text:
        print(chunk.text, end="", flush=True)

print("\n\n--- done ---")

That is how ChatGPT and Gemini feel fast. Nothing clever — they simply stream.
One changed method name (`generate_content_stream`) and your app does too.

## Cell 5 — Multi-turn chat

**A plain API call has no memory.** Every request starts from nothing.

The `chats` helper carries the history for you.

In [ ]:
chat = client.chats.create(model=MODEL)

r1 = chat.send_message("My name is Abebe and I teach at the University of Gondar.")
print("1:", r1.text[:200])
print()

r2 = chat.send_message("Where do I teach?")
print("2:", r2.text[:200])
print()

print("--- what the model actually has in memory ---")
for m in chat.get_history():
    print(f"  {m.role}: {m.parts[0].text[:70]}")

> ### The hidden cost of chat
> Every turn re-sends the **entire** conversation. Turn 20 costs roughly twenty times
> turn 1 in input tokens.
>
> Long chats get expensive quietly. Real applications keep only the last N turns, or ask
> the model to summarise the conversation and start fresh from that summary.

## Cell 6 — Structured JSON output

Prose is for humans. If your **program** must read the answer, ask for JSON.

In [ ]:
import json

config = types.GenerateContentConfig(
    response_mime_type="application/json",
    system_instruction=(
        "Return a JSON object with exactly these keys: "
        "name (string), region (string), founded_year (number), "
        "known_for (array of strings)."),
)

r = client.models.generate_content(
    model=MODEL,
    contents="Give me facts about the city of Gondar, Ethiopia.",
    config=config,
)

data = json.loads(r.text)        # a real Python dictionary

print(json.dumps(data, indent=2, ensure_ascii=False))
print()
print("Access it like normal data:")
print("  name   :", data.get("name"))
print("  region :", data.get("region"))

**This is the difference between a demo and a product.** A product needs answers in a
shape the next line of code can use — not a paragraph you have to clean up with string
searching.

## Cell 7 — Batch processing with rate-limit awareness

A realistic task: classify many items. This is where you hit rate limits.

In [ ]:
import time

feedback = [
    "The training was excellent and very clear.",
    "ክፍሉ በጣም ሞቃት ነበር።",
    "I did not understand the section on fine-tuning.",
    "Best course I have attended this year.",
    "The internet kept disconnecting.",
]

config = types.GenerateContentConfig(
    response_mime_type="application/json",
    system_instruction=(
        "Classify the feedback. Return JSON with keys: "
        "sentiment (positive/negative/neutral), topic (string), language (string)."))

results = []
for i, text in enumerate(feedback, 1):
    try:
        r = client.models.generate_content(model=MODEL, contents=text, config=config)
        item = json.loads(r.text)
        item["text"] = text[:40]
        results.append(item)
        print(f"{i}. {item['sentiment']:<9} {item['language']:<10} {item['topic'][:35]}")
    except Exception as e:
        print(f"{i}. ERROR: {str(e)[:80]}")

    time.sleep(1)        # be polite - stay under the per-minute limit

print(f"\nProcessed {len(results)} of {len(feedback)} items")

> ### Why `time.sleep(1)`
> Free tiers cap requests **per minute**. Firing a loop as fast as possible will hit
> that cap within seconds. One second between calls keeps you comfortably inside it.
>
> Never retry in a tight loop — that burns the very quota you are waiting for.

## Cell 8 — Check your remaining budget

There is no API call that reports your remaining quota, so track it yourself.

In [ ]:
class UsageTracker:
    def __init__(self, daily_limit=200):
        self.calls = 0
        self.input_tokens = 0
        self.output_tokens = 0
        self.daily_limit = daily_limit

    def record(self, response):
        u = response.usage_metadata
        self.calls += 1
        self.input_tokens += u.prompt_token_count
        self.output_tokens += u.candidates_token_count

    def report(self):
        print(f"Calls made      : {self.calls} / ~{self.daily_limit} assumed daily")
        print(f"Input tokens    : {self.input_tokens:,}")
        print(f"Output tokens   : {self.output_tokens:,}")


tracker = UsageTracker()

for q in ["What is AI?", "What is an API?", "What is a token?"]:
    r = client.models.generate_content(model=MODEL, contents=q)
    tracker.record(r)

tracker.report()
print()
print("Actual current limits: ai.google.dev/gemini-api/docs/rate-limits")

---

## Checklist

- [ ] I set a system instruction and saw the model obey it
- [ ] I understand that `temperature` is deprecated on Gemini 3.x
- [ ] I controlled style using **instructions in words** instead
- [ ] I streamed a response
- [ ] I built a chat that remembered earlier turns
- [ ] I got valid JSON and parsed it with `json.loads()`
- [ ] I processed a batch without hitting the rate limit

**Next:** Session 3 — build a real assistant.